In [1]:
!wget https://huggingface.co/datasets/Jgmorenof/teaching_tools_2025/resolve/main/chef-douvre.zip

--2026-03-10 13:57:07--  https://huggingface.co/datasets/Jgmorenof/teaching_tools_2025/resolve/main/chef-douvre.zip
Resolving huggingface.co (huggingface.co)... 18.239.50.103, 18.239.50.49, 18.239.50.80, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.103|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/68a6cbb4eefddf148411cee1/54a51d564f259297a0705d4873bb2975d6589c79119ea8729eb93788a856ab4d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27chef-douvre.zip%3B+filename%3D%22chef-douvre.zip%22%3B&response-content-type=application%2Fzip&Expires=1773154627&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzczMTU0NjI3fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjhhNmNiYjRlZWZkZGYxNDg0MTFjZWUxLzU0YTUxZDU2NGYyNTkyOTdhMDcwNWQ0ODczYmIyOTc1ZDY1ODljNzkxMTllYTg3MjllYjkzNzg4YTg1NmFiNGRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29ud

In [2]:
!unzip chef-douvre.zip

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/urbi.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/ncl.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/cplint.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/r.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/_usd_builtins.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/comal.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/gsql.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/freefem.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/codeql.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/pygments/lexers/modula2.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packag

In [5]:
# ==========================================================
# 1. INSTALLATION, IMPORTS & REPRODUCTIBILITÉ
# ==========================================================
!pip install -q lxml pandas numpy scikit-learn sentence-transformers lightgbm networkx tqdm

import os, re, glob, random
import numpy as np
import pandas as pd
from lxml import etree
from collections import Counter
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score, v_measure_score)
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import warnings

# --- Reproductibilité Stricte ---
# Fixe le hasard pour que les résultats soient toujours les mêmes
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
warnings.filterwarnings('ignore')
print("✅ Environnement prêt. Le hasard a été fixé pour la reproductibilité.")

✅ Environnement prêt. Le hasard a été fixé pour la reproductibilité.


In [7]:
# ==========================================================
# 2. TÉLÉCHARGEMENT DU DATASET
# ==========================================================
URL = "https://huggingface.co/datasets/Jgmorenof/teaching_tools_2025/resolve/main/chef-douvre.zip"
if not os.path.exists("chef-douvre"):
    print("📥 Téléchargement des données (3Go)...")
    !wget -q -O chef-douvre.zip {URL}
    !unzip -q chef-douvre.zip
    print("✅ Données prêtes.")
else:
    print("✅ Données déjà présentes.")

✅ Données déjà présentes.


In [8]:
# ==========================================================
# 3. CONFIGURATION ET PARAMÈTRES
# ==========================================================
XML_FOLDER  = "/content/chef-douvre/AS_TrainingSet_BnF_NewsEye_v2"
THRESHOLD   = 0.85    # Seuil de confiance pour créer un lien dans le graphe
K_NEIGHBORS = 20      # Nombre de voisins à considérer pour chaque bloc
N_FILES     = 183     # Nombre de fichiers à traiter (183 pour le dataset complet)

LGBM_PARAMS = dict(
    n_estimators      = 500,
    learning_rate     = 0.04,
    num_leaves        = 63,
    min_child_samples = 20,
    class_weight      = 'balanced',
    random_state      = 42, # Garanti par set_seed
    verbose           = -1,
)
print("✅ Paramètres configurés.")

✅ Paramètres configurés.


In [ ]:
# ==========================================================
# 4. FONCTIONS DE PARSING ET D'EMBEDDING
# ==========================================================

# (Ici on place les fonctions de parsing : detect_ns, get_coords, etc.)
def detect_ns(root):
    tag = root.tag
    if tag.startswith('{'): return {'ns': tag[1:tag.index('}')]}
    return {}
def find_any(el, tag, ns): return el.find(f".//ns:{tag}", ns) if ns else el.find(f".//{tag}")
def findall_any(el, tag, ns): return el.findall(f'.//ns:{tag}', ns) if ns else el.findall(f'.//{tag}')
def findtext_any(el, tag, ns, default=""):
    e = find_any(el, tag, ns)
    return e.text.strip() if e is not None and e.text else default
def get_coords(pts_str):
    pts =[list(map(int, p.split(','))) for p in pts_str.split()]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    return min(xs), min(ys), max(xs)-min(xs), max(ys)-min(ys)
def extract_gt_id(custom_attr):
    m = re.search(r'structure\s*{\s*id:\s*(a\d+)', custom_attr)
    return m.group(1).strip() if m else 'NO_ID'

def load_and_embed(xml_folder, limit=None):
    xml_files = sorted(glob.glob(os.path.join(xml_folder, "*.xml")))
    if limit: xml_files = xml_files[:limit]

    data = []
    print(f"📂 Parsing de {len(xml_files)} fichiers XML...")
    for path in tqdm(xml_files, desc="Parsing XML"):
        try:
            tree, root = etree.parse(path), etree.parse(path).getroot()
            NS = detect_ns(root)
            page = find_any(root, 'Page', NS)
            if page is None: continue
            p_w, p_h = float(page.get("imageWidth", 1)), float(page.get("imageHeight", 1))
            fname = os.path.basename(path)
            for r in findall_any(root, 'TextRegion', NS):
                lines = findall_any(r, 'TextLine', NS)
                text = " ".join([findtext_any(l,'Unicode',NS) for l in lines]).strip()
                if not text: continue
                ids = [extract_gt_id(l.get('custom','')) for l in lines]
                true_id = Counter([i for i in ids if i!='NO_ID']).most_common(1)[0][0] if any(i!='NO_ID' for i in ids) else 'NO_ID'
                ce = find_any(r, 'Coords', NS)
                if ce is None or not ce.get('points',''): continue
                x, y, w, h = get_coords(ce.get('points',''))
                data.append({
                    'filename': fname, 'id_region': r.get('id'), 'text': text,
                    'n_lines': len(lines), 'article_id': true_id,
                    'article_id_global': f"{fname}__{true_id}" if true_id!='NO_ID' else 'NO_ID',
                    'cx': (x+w/2)/p_w, 'cy': (y+h/2)/p_h, 'w': w/p_w, 'h': h/p_h
                })
        except: continue
    df = pd.DataFrame(data)
    print(f"✅ {len(df)} blocs extraits.")

    print("\n📐 Calcul des embeddings sémantiques (BGE-M3)...")
    sbert = SentenceTransformer('BAAI/bge-m3')
    text_emb = sbert.encode(df['text'].tolist(), show_progress_bar=True, normalize_embeddings=True)
    print(f"✅ {len(text_emb)} vecteurs sémantiques créés.")
    return df, text_emb

df, text_emb = load_and_embed(XML_FOLDER, limit=N_FILES)

📂 Parsing de 183 fichiers XML...


Parsing XML:   0%|          | 0/183 [00:00<?, ?it/s]

✅ 62343 blocs extraits.

📐 Calcul des embeddings sémantiques (BGE-M3)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
# ==========================================================
# 5. FEATURE ENGINEERING
# ==========================================================
def det_col(cx, n=6): return min(int(cx*n), n-1)
def pair_features(ri, rj, ei, ej):
    cos = np.dot(ei, ej) # Déjà normalisé
    dcx, dcy = ri['cx']-rj['cx'], ri['cy']-rj['cy']
    ci, cj = det_col(ri['cx']), det_col(rj['cx'])
    dist = np.hypot(dcx, dcy)
    same_col = int(abs(dcx) < 0.12)
    return np.array([
        cos, abs(dcx), abs(dcy), dcx, dcy, dist,
        abs(ci-cj), float(ci==cj), float(abs(ci-cj)==1),
        ri['w'], rj['w'], ri['h'], rj['h'], abs(ri['w']-rj['w']), abs(ri['h']-rj['h']),
        ri['w']/max(rj['w'],1e-6), ri['h']/max(rj['h'],1e-6), ri['h']/max(ri['w'],1e-6), rj['h']/max(rj['w'],1e-6),
        same_col, int(dcy>0) if same_col else int(dcx>0), float(rj['cy']>ri['cy']), float(rj['cx']>ri['cx']),
        ri['w']*ri['h'], rj['w']*rj['h'], (ri['w']*ri['h'])/max(rj['w']*rj['h'],1e-9),
        ri['n_lines'], rj['n_lines'], ri['n_lines']/max(rj['n_lines'],1e-6), (ri['cy']+rj['cy'])/2
    ])

print("🔗 Construction du jeu de données par paires...")
page_data = {}
for fname, dfp in tqdm(df.groupby('filename'), desc="Couplage par pages"):
    idx, emb_p = dfp.index.tolist(), text_emb[dfp.index.tolist()]
    dfp_r = dfp.reset_index(drop=True)
    coords = dfp_r[['cx','cy']].values
    pairs, feats, labels = [], [], []
    for ii in range(len(dfp_r)):
        dists = np.linalg.norm(coords - coords[ii], axis=1)
        for jj in np.argsort(dists)[1:K_NEIGHBORS+1]:
            if ii >= jj: continue
            ri, rj = dfp_r.iloc[ii], dfp_r.iloc[jj]
            feat = pair_features(ri, rj, emb_p[ii], emb_p[jj])
            same = int(ri['article_id'] == rj['article_id'] and ri['article_id'] != 'NO_ID')
            pairs.append((idx[ii], idx[jj]))
            feats.append(feat)
            labels.append(same)
    page_data[fname] = {'pairs': pairs, 'feats': np.array(feats), 'labels': np.array(labels)}

total = sum(len(v['pairs']) for v in page_data.values())
pos = sum(v['labels'].sum() for v in page_data.values())
print(f"✅ {total} paires créées ({pos*100/total:.1f}% positives).")

In [ ]:
# ==========================================================
# 6. ENTRAÎNEMENT & ÉVALUATION (SPLIT STRICT)
# ==========================================================
all_pages = list(page_data.keys())
train_pages, test_pages = train_test_split(all_pages, test_size=0.20, random_state=42)

print(f"📚 Entraînement sur {len(train_pages)} pages...")
X_tr = np.vstack([page_data[p]['feats'] for p in train_pages if len(page_data[p]['feats']) > 0])
y_tr = np.concatenate([page_data[p]['labels'] for p in train_pages])
clf = LGBMClassifier(**LGBM_PARAMS)
clf.fit(X_tr, y_tr)
print("✅ Modèle entraîné.")

print(f"\n🎯 Évaluation sur les {len(test_pages)} pages de test...")
all_pred_labels, ari_pages = {}, []
for test_page in tqdm(test_pages, desc="Évaluation Test Set"):
    pd_test = page_data[test_page]
    if len(pd_test['feats']) == 0: continue
    probs = clf.predict_proba(pd_test['feats'])[:,1]
    dfp = df[df['filename'] == test_page]
    G = nx.Graph()
    G.add_nodes_from(dfp.index)
    for (i, j), p in zip(pd_test['pairs'], probs):
        if p > THRESHOLD:
            ri, rj = df.loc[i], df.loc[j]
            dy = rj['cy'] - ri['cy']
            if dy > 0.15 and p < 0.95: continue
            if rj['n_lines'] < 2 and p < 0.90: continue
            G.add_edge(i, j)
    pred = {}
    for cid, comp in enumerate(nx.connected_components(G)):
        for node in comp:
            pred[node] = cid
            all_pred_labels[node] = f"{test_page}_c{cid}"
    yt, yp = dfp['article_id'].astype(str), np.array([str(pred.get(i,-1)) for i in dfp.index])
    if len(np.unique(yt)) > 1: ari_pages.append(adjusted_rand_score(yt, yp))

print("✅ Évaluation terminée.")

In [ ]:
# ==========================================================
# 7. RAPPORT DE PERFORMANCE
# ==========================================================
df['predicted_cluster'] = [all_pred_labels.get(i, "NOT_IN_TEST") for i in df.index]
df_test = df[df['filename'].isin(test_pages)]
df_g = df_test[df_test['article_id_global'] != 'NO_ID']
yt_g, yp_g = df_g['article_id_global'].astype(str), df_g['predicted_cluster'].astype(str)

print("\n" + "═"*60)
print(" 📊  RAPPORT DE PERFORMANCE SUR LE SET DE TEST (INVIOLÉ)")
print("═"*60)
print(f"  🏆 ARI Moyen par Page : {np.mean(ari_pages):.4f} ± {np.std(ari_pages):.3f}")
print(f"  - ARI Global         : {adjusted_rand_score(yt_g, yp_g):.4f}")
print(f"  - V-Measure Global   : {v_measure_score(yt_g, yp_g):.4f}")
print("─"*60)
print(f"  📚 Articles Réels     : {df_g['article_id'].nunique()}")
print(f"  🎯 Clusters Prédits    : {df_g['predicted_cluster'].nunique()}")
print("═"*60)

In [ ]:
# ==========================================================
# 8. ANALYSE VISUELLE
# ==========================================================
def visualize_page(dataframe, filename, img_folder):
    df_page = dataframe[dataframe['filename'] == filename].copy()
    if df_page.empty: return print(f"'{filename}' non trouvé.")
    path_img = os.path.join(img_folder, filename.replace(".xml", ".jpg"))
    if not os.path.exists(path_img): return print(f"Image introuvable: {path_img}")
    img = Image.open(path_img)
    img_w, img_h = img.size
    df_eval = df_page[df_page['article_id'] != 'NO_ID']
    ari = adjusted_rand_score(df_eval['article_id'], df_eval['predicted_cluster']) if len(df_eval)>1 else 0

    fig, ax = plt.subplots(1, 2, figsize=(24, 16))
    cmap = plt.get_cmap('tab20')
    ax[0].imshow(img, alpha=0.7); ax[1].imshow(img, alpha=0.7)
    ax[0].set_title(f"🟢 VÉRITÉ TERRAIN ({df_eval['article_id'].nunique()} Articles)", fontsize=18)
    ax[1].set_title(f"🔴 PRÉDICTION (ARI: {ari:.3f})", fontsize=18)
    ax[0].axis('off'); ax[1].axis('off')

    unique_gt, unique_pred = df_eval['article_id'].unique().tolist(), df_eval['predicted_cluster'].unique().tolist()
    for _, row in df_eval.iterrows():
        # GT
        c_gt = cmap(unique_gt.index(row['article_id']) % 20)
        w, h = row['w']*img_w, row['h']*img_h
        x, y = (row['cx']*img_w)-(w/2), (row['cy']*img_h)-(h/2)
        ax[0].add_patch(patches.Rectangle((x,y),w,h, ec='black', fc=(c_gt[0],c_gt[1],c_gt[2],0.5)))
        # PRED
        c_pred = cmap(unique_pred.index(row['predicted_cluster']) % 20)
        ax[1].add_patch(patches.Rectangle((x,y),w,h, ec='black', fc=(c_pred[0],c_pred[1],c_pred[2],0.5)))
    plt.tight_layout()
    plt.show()

# --- Visualisation des Cas d'Intérêt ---
page_scores = pd.DataFrame({'filename': test_pages, 'ari': ari_pages}).sort_values('ari', ascending=False)

print("\n✅ Analyse d'un cas réussi (meilleur ARI du Test Set):")
visualize_page(df, page_scores.iloc[0]['filename'], XML_FOLDER)

print("\n❌ Analyse d'un cas difficile (pire ARI du Test Set):")
visualize_page(df, page_scores.iloc[-1]['filename'], XML_FOLDER)```
---
### CELLULE 9 : Export Final

<div class="markdown-google-sans">

## Étape 8 : Exportation
Les résultats finaux, incluant les IDs de cluster prédits, sont sauvegardés dans un fichier CSV pour une utilisation future.

</div>

```python
# ==========================================================
# 9. EXPORTATION DES RÉSULTATS
# ==========================================================
output_file = "predictions_finales.csv"
export_df = df[df['filename'].isin(test_pages)][['filename', 'id_region', 'text', 'article_id', 'predicted_cluster']]
export_df.to_csv(output_file, index=False)
print(f"✅ Résultats du Test Set sauvegardés dans : {output_file}")